In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score,roc_auc_score
from collections import defaultdict

In [2]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.xls')

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [5]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

In [6]:
# df['HasFamily'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).map({True: 'Yes', False: 'No'})

In [7]:
#df['Is_streaming'] = ((df['StreamingMovies'] == 'Yes') | (df['StreamingTV'] == 'Yes')).map({True:'Yes',False:'No'})

df['Is_streaming'] = np.where(
    (df['StreamingMovies'] == 'Nointernetservice') | (df['StreamingTV'] == 'Nointernetservice'), 'Nointernetservice',
        np.where(
            (df['StreamingMovies'] == 'Yes') | (df['StreamingTV'] == 'Yes'), 'Yes', 'No'))

In [ ]:
# df['Monthly_charges_flag'] = np.where((df['MonthlyCharges']<=40),'LowMonthlyCharges',
#                                       np.where((df['MonthlyCharges']>40) & (df['MonthlyCharges'] <= 70),'MediumMonthlyCharges','HighMonthlyCharges'))

In [8]:
#df['Online_backup_security'] = ((df['OnlineBackup'] == 'Yes') | (df['OnlineSecurity'] == 'Yes')).map({True:"Yes",False:'No'})
# df['Online_backup_security'] = np.where(
#     (df['OnlineSecurity'] == 'Nointernetservice') | (df['OnlineBackup'] == 'Nointernetservice'), 'Nointernetservice',
#         np.where(
#             (df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes'), 'Yes', 'No'))

In [ ]:
drop_columns = ['customerID','StreamingMovies','StreamingTV']
for col in drop_columns:
    if col in df.columns:
        df.drop(columns=[col],inplace=True)
#df.drop(columns=['customerID','Partner','Dependents','StreamingMovies','StreamingTV'],inplace=True)
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn',
       'Is_streaming', 'Monthly_charges_flag'],
      dtype='str')

In [44]:
#df['PaymentMethod'] = (df['PaymentMethod'] == 'Electroniccheck').map({True:'Electroniccheck',False:'Other'})

In [45]:
X = df.drop(columns=['Churn'])
y = df['Churn']
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,TotalCharges,Is_streaming,Monthly_charges_flag
0,Female,0,Yes,No,1,No,Nophoneservice,DSL,No,Yes,No,No,Monthtomonth,Yes,Electroniccheck,29.85,No,LowMonthlyCharges
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,Oneyear,No,Other,1889.50,No,MediumMonthlyCharges
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,Monthtomonth,Yes,Other,108.15,No,MediumMonthlyCharges
3,Male,0,No,No,45,No,Nophoneservice,DSL,Yes,No,Yes,Yes,Oneyear,No,Other,1840.75,No,MediumMonthlyCharges
4,Female,0,No,No,2,Yes,No,Fiberoptic,No,No,No,No,Monthtomonth,Yes,Electroniccheck,151.65,No,HighMonthlyCharges
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Oneyear,Yes,Other,1990.50,Yes,HighMonthlyCharges
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiberoptic,No,Yes,Yes,No,Oneyear,Yes,Other,7362.90,Yes,HighMonthlyCharges
7040,Female,0,Yes,Yes,11,No,Nophoneservice,DSL,Yes,No,No,No,Monthtomonth,Yes,Electroniccheck,346.45,No,LowMonthlyCharges
7041,Male,1,Yes,No,4,Yes,Yes,Fiberoptic,No,No,No,No,Monthtomonth,Yes,Other,306.60,No,HighMonthlyCharges


In [46]:
numerical_feature = X.select_dtypes(exclude="str").columns
categorical_feature = X.select_dtypes(include="str").columns
categorical_feature

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'Is_streaming', 'Monthly_charges_flag'],
      dtype='str')

In [47]:
numerical_feature

Index(['SeniorCitizen', 'tenure', 'TotalCharges'], dtype='str')

In [48]:
for col in categorical_feature:
    df[col] =  df[col].str.replace(' ', '')
    df[col] =  df[col].str.replace('-', '')

In [49]:
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,TotalCharges,Is_streaming,Monthly_charges_flag
0,Female,0,Yes,No,1,No,Nophoneservice,DSL,No,Yes,No,No,Monthtomonth,Yes,Electroniccheck,29.85,No,LowMonthlyCharges
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,Oneyear,No,Other,1889.50,No,MediumMonthlyCharges
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,Monthtomonth,Yes,Other,108.15,No,MediumMonthlyCharges
3,Male,0,No,No,45,No,Nophoneservice,DSL,Yes,No,Yes,Yes,Oneyear,No,Other,1840.75,No,MediumMonthlyCharges
4,Female,0,No,No,2,Yes,No,Fiberoptic,No,No,No,No,Monthtomonth,Yes,Electroniccheck,151.65,No,HighMonthlyCharges


In [50]:
preprocessor = ColumnTransformer([
    ('OHE',OneHotEncoder(drop='first'),categorical_feature),
    ('SC',StandardScaler(),numerical_feature)
])

In [51]:
le = LabelEncoder()

y = le.fit_transform(y)
y

array([0, 0, 1, ..., 0, 1, 0], shape=(7043,))

In [52]:
X = preprocessor.fit_transform(X)


In [53]:
X

array([[ 0.        ,  1.        ,  0.        , ..., -0.43991649,
        -1.27744458, -0.99497138],
       [ 1.        ,  0.        ,  0.        , ..., -0.43991649,
         0.06632742, -0.17387565],
       [ 1.        ,  0.        ,  0.        , ..., -0.43991649,
        -1.23672422, -0.96039939],
       ...,
       [ 0.        ,  1.        ,  1.        , ..., -0.43991649,
        -0.87024095, -0.85518222],
       [ 1.        ,  1.        ,  0.        , ...,  2.27315869,
        -1.15528349, -0.87277729],
       [ 1.        ,  0.        ,  0.        , ..., -0.43991649,
         1.36937906,  2.01391739]], shape=(7043, 26))

In [54]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
print(X_train)
print(y_train)

[[ 1.          1.          1.         ... -0.43991649  0.88073469
   0.65642602]
 [ 1.          0.          0.         ... -0.43991649 -1.27744458
  -0.97258569]
 [ 1.          0.          0.         ... -0.43991649 -0.78880022
  -0.89350724]
 ...
 [ 1.          1.          1.         ... -0.43991649 -0.82952058
  -0.87302013]
 [ 1.          0.          0.         ...  2.27315869 -0.82952058
  -0.47824601]
 [ 1.          0.          0.         ... -0.43991649 -0.25943549
  -0.80623836]]
[0 0 0 ... 0 1 0]


In [55]:
models = {
                "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
                "Random Forest": RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42),
                "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
                "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42),
                "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
                "Gradient Boosting": GradientBoostingClassifier(random_state=42),
                "SVM": SVC(class_weight='balanced', probability=True, random_state=42),
                "KNN": KNeighborsClassifier(n_neighbors=5)
            }

In [56]:
if 'report' not in locals():
    report = defaultdict(list)


In [57]:
for model_name,model in models.items():

    model.fit(X_train,y_train)

    predict = model.predict(X_test)

    roc_auc = roc_auc_score(y_test,predict)*100
    f1__score = f1_score(y_test,predict)*100

    report[model_name].append((roc_auc, f1__score))

    #report[model_name] = roc_auc,f1__score

[LightGBM] [Info] Number of positive: 1295, number of negative: 3635
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001146 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 378
[LightGBM] [Info] Number of data points in the train set: 4930, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [58]:
report

defaultdict(list,
            {'Logistic Regression': [(78.41758868716506, 65.36124240378123),
              (78.26550341526809, 65.18218623481782),
              (78.26550341526809, 65.18218623481782),
              (78.20052615730835, 65.09433962264151)],
             'Random Forest': [(73.61012060413003, 61.17065127782357),
              (73.49193897118587, 60.94771241830066),
              (73.49193897118587, 60.94771241830066),
              (72.65476247076589, 59.80392156862745)],
             'XGBoost': [(70.51045635769641, 57.33590733590733),
              (70.84853054044325, 57.8544061302682),
              (70.84853054044325, 57.8544061302682),
              (69.32909283144627, 55.39906103286385)],
             'LightGBM': [(75.37961887555383, 62.601028655400434),
              (76.48847729078796, 63.970588235294116),
              (76.48847729078796, 63.970588235294116),
              (77.1010634082949, 64.50683945284376)],
             'CatBoost': [(71.18235969327111, 58.48

In [59]:

# for model, scores in sorted(report.items(), key=lambda item: item[1], reverse=True):
#     print(f"{model} -> roc_auc_score: {scores[0]:.4f}")
#     print(f"{model} -> f1_score: {scores[1]}")
#     print("\n")

table_data = []
for model_name, scores in report.items():
    # Fetch old and new scores safely
    old_roc, old_f1 = scores[0] if len(scores) >= 2 else (None, None)
    new_roc, new_f1 = scores[-1] if scores else (None, None)
    
    table_data.append({
        'Model Name': model_name,
        'Old ROC-AUC': old_roc,
        'New ROC-AUC': new_roc,
        'Old F1': old_f1,
        'New F1': new_f1
    })

# Convert to DataFrame and display
df_results = pd.DataFrame(table_data)
print(df_results)  # Or just type 'df_results' if in a Jupyter notebook cell


            Model Name  Old ROC-AUC  New ROC-AUC     Old F1     New F1
0  Logistic Regression    78.417589    78.200526  65.361242  65.094340
1        Random Forest    73.610121    72.654762  61.170651  59.803922
2              XGBoost    70.510456    69.329093  57.335907  55.399061
3             LightGBM    75.379619    77.101063  62.601029  64.506839
4             CatBoost    71.182360    71.228036  58.488714  58.494208
5    Gradient Boosting    71.213433    71.858508  58.589871  59.516908
6                  SVM    78.152020    78.249486  65.328720  65.464632
7                  KNN    69.584304    70.487873  55.746606  57.091562


In [60]:

# 1. Format the scores as strings "ROC / F1" for clean rows
formatted_report = {}
for model_name, scores in report.items():
    formatted_report[model_name] = [f"{roc:.4f} / {f1:.4f}" for roc, f1 in scores]

# 2. Create the DataFrame (this automatically sets model names as column headers)
# We use pd.DataFrame.from_dict with orient='columns' (default)
df_transposed = pd.DataFrame.from_dict(formatted_report, orient='columns')

# 3. Label the row index to represent the experiment runs
df_transposed.index = [f"Run {i+1}" for i in range(len(df_transposed))]

# Display the DataFrame
df_transposed


,Logistic Regression,Random Forest,XGBoost,LightGBM,CatBoost,Gradient Boosting,SVM,KNN
Run 1,78.4176 / 65.3612,73.6101 / 61.1707,70.5105 / 57.3359,75.3796 / 62.6010,71.1824 / 58.4887,71.2134 / 58.5899,78.1520 / 65.3287,69.5843 / 55.7466
Run 2,78.2655 / 65.1822,73.4919 / 60.9477,70.8485 / 57.8544,76.4885 / 63.9706,71.1188 / 58.3333,71.3123 / 58.7192,78.2495 / 65.4646,69.9210 / 56.2613
Run 3,78.2655 / 65.1822,73.4919 / 60.9477,70.8485 / 57.8544,76.4885 / 63.9706,71.1188 / 58.3333,71.3123 / 58.7192,78.2495 / 65.4646,69.9210 / 56.2613
Run 4,78.2005 / 65.0943,72.6548 / 59.8039,69.3291 / 55.3991,77.1011 / 64.5068,71.2280 / 58.4942,71.8585 / 59.5169,78.2495 / 65.4646,70.4879 / 57.0916
